# Adult Census Income Prediction
## ML Classification Assignment

**Dataset:** UCI Adult Census Income  
**Goal:** Predict whether an individual earns >$50K/year  
**Tasks:** Dataset Understanding → Data Cleaning → Feature Engineering → Model Building → Performance Evaluation

---

## 📦 Imports & Setup

In [ ]:
# =============================================================================
# Adult Census Income Prediction — ML Classification Assignment
# Dataset: UCI Adult Census Income Dataset
# Goal: Predict whether an individual earns >$50K/year
# =============================================================================

## Task 1 — Dataset Understanding (10 Marks)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)

import warnings
warnings.filterwarnings('ignore')

# Setting a consistent plot style throughout the notebook
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 100

### 1.2 — Data Types & Statistical Summary

In [ ]:
# Loading directly from UCI — no manual download needed
# The dataset uses comma+space as separator, hence sep=', '
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]

df = pd.read_csv(url, names=columns, sep=', ', engine='python')

print("=" * 55)
print("         ADULT CENSUS INCOME DATASET — OVERVIEW")
print("=" * 55)
print(f"\n  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")
print(f"\n  Features breakdown:")
print(f"    → Numerical  : {df.select_dtypes(include='number').shape[1]}")
print(f"    → Categorical: {df.select_dtypes(include='object').shape[1]}")
print("\nFirst 5 rows:")
print(df.head())

### 1.3 — Target Distribution & EDA Plots

In [ ]:
print("\nColumn Data Types:")
print(df.dtypes)

print("\nStatistical Summary (numerical features):")
print(df.describe().round(2))

# fnlwgt is a census weight column — not directly useful for prediction
# education and education_num carry overlapping info (one is a label, other is ordinal)
# These observations will guide feature engineering decisions later

## Task 2 — Data Cleaning (20 Marks)

In [ ]:
print("\nTarget class distribution:")
print(df['income'].value_counts())
print(f"\nClass imbalance ratio: {round(df['income'].value_counts()[0] / df['income'].value_counts()[1], 2)}:1")
# The dataset is imbalanced (~3:1) — worth keeping in mind during evaluation
# F1 score will be more reliable than raw accuracy here

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis — Adult Census Income', fontsize=15, fontweight='bold')

# Plot 1: Income class distribution
df['income'].value_counts().plot(
    kind='bar', ax=axes[0, 0],
    color=['#4C72B0', '#DD8452'], edgecolor='black'
)
axes[0, 0].set_title('Target: Income Distribution')
axes[0, 0].set_xlabel('Income Class')
axes[0, 0].set_ylabel('Count')
axes[0, 0].tick_params(axis='x', rotation=0)
for p in axes[0, 0].patches:
    axes[0, 0].annotate(f'{int(p.get_height()):,}',
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10)

# Plot 2: Age distribution by income — gives a sense of who earns more
df_temp = df.copy()
df_temp['income_label'] = df_temp['income']
df_temp[df_temp['income'] == '<=50K']['age'].plot(
    kind='hist', ax=axes[0, 1], alpha=0.6, bins=30, label='<=50K', color='#4C72B0'
)
df_temp[df_temp['income'] == '>50K']['age'].plot(
    kind='hist', ax=axes[0, 1], alpha=0.6, bins=30, label='>50K', color='#DD8452'
)
axes[0, 1].set_title('Age Distribution by Income Class')
axes[0, 1].set_xlabel('Age')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].legend()

# Plot 3: Income split by sex — quick look at demographic patterns
sex_income = df.groupby(['sex', 'income']).size().unstack()
sex_income.plot(kind='bar', ax=axes[1, 0], color=['#4C72B0', '#DD8452'], edgecolor='black')
axes[1, 0].set_title('Income Distribution by Sex')
axes[1, 0].set_xlabel('Sex')
axes[1, 0].set_ylabel('Count')
axes[1, 0].tick_params(axis='x', rotation=0)
axes[1, 0].legend(title='Income')

# Plot 4: Hours per week distribution — checking for work pattern patterns
df['hours_per_week'].plot(
    kind='hist', ax=axes[1, 1], bins=30, color='#55A868', edgecolor='black', alpha=0.8
)
axes[1, 1].axvline(df['hours_per_week'].mean(), color='red', linestyle='--', label=f"Mean: {df['hours_per_week'].mean():.1f} hrs")
axes[1, 1].set_title('Hours Per Week Distribution')
axes[1, 1].set_xlabel('Hours Per Week')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('eda_overview.png', bbox_inches='tight')
plt.show()
print("EDA plots saved.")

## Task 3 — Feature Engineering (15 Marks)

In [ ]:
print("=" * 45)
print("         TASK 2: DATA CLEANING")
print("=" * 45)

# This dataset uses '?' as a placeholder for missing values
# Need to convert those to NaN before any imputation
print("\nMissing values encoded as '?' per column:")
print((df == '?').sum()[lambda x: x > 0])

df.replace('?', np.nan, inplace=True)

print("\nNull counts after replacing '?' with NaN:")
print(df.isnull().sum()[lambda x: x > 0])

# Imputing with mode — appropriate for categorical columns like workclass, occupation
# Since these are nominal, mean/median don't apply
for col in df.columns:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  → Filled '{col}' with mode: '{mode_val}'")

print(f"\nDuplicate rows found: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Duplicates removed. Clean dataset shape: {df.shape}")

# ── Outlier Detection using IQR ──────────────────────────────────────────────
# Checking numerical columns for extreme outliers that could skew model training

print("\nOutlier check using IQR method (numerical columns):")
numerical_cols = ['age', 'fnlwgt', 'capital_gain', 'capital_loss', 'hours_per_week', 'education_num']

outlier_summary = []
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    outlier_summary.append({'Feature': col, 'Lower Bound': round(lower, 2),
                            'Upper Bound': round(upper, 2), 'Outlier Count': outliers})

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

# capital_gain and capital_loss have heavy right skew — many zeros with a few extreme values
# Keeping them as-is since extreme values here are economically meaningful, not errors

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Outlier Visualization — Capital Gain & Loss', fontsize=13)
df.boxplot(column='capital_gain', ax=axes[0])
axes[0].set_title('Capital Gain')
df.boxplot(column='capital_loss', ax=axes[1])
axes[1].set_title('Capital Loss')
plt.tight_layout()
plt.savefig('outliers_boxplot.png', bbox_inches='tight')
plt.show()

### 3.2 — Train/Test Split & Feature Scaling

In [ ]:
print("=" * 45)
print("       TASK 3: FEATURE ENGINEERING")
print("=" * 45)

# Encoding the target first
df['income'] = df['income'].str.strip()
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})
print(f"\nTarget encoded → <=50K: 0, >50K: 1")

# Dropping 'education' since 'education_num' already captures it numerically
# Dropping 'fnlwgt' — it's a census sampling weight, not a predictive feature
df.drop(columns=['education', 'fnlwgt'], inplace=True)
print("Dropped: 'education' (redundant with education_num), 'fnlwgt' (sampling weight)")

# Label encoding all remaining categorical columns
le = LabelEncoder()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"\nEncoding categorical columns: {cat_cols}")

for col in cat_cols:
    df[col] = df[col].str.strip()
    df[col] = le.fit_transform(df[col])

print("\nSample after encoding:")
print(df.head(3))

# Correlation heatmap — checking multicollinearity between features
plt.figure(figsize=(13, 9))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # only lower triangle
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', linewidths=0.5, vmin=-1, vmax=1
)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

## Task 4 — Model Building (30 Marks)

In [ ]:
X = df.drop('income', axis=1)
y = df['income']

# 80/20 split — standard for this dataset size
# random_state=42 for reproducibility across runs
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
    # stratify=y ensures both splits have the same class ratio as the full dataset
)

print(f"Training set : {X_train.shape[0]:,} samples")
print(f"Testing set  : {X_test.shape[0]:,} samples")
print(f"Class ratio (train): {y_train.value_counts(normalize=True).round(3).to_dict()}")

# Standard scaling — critical for distance-based models (KNN, SVM)
# Tree-based models don't need it but it doesn't hurt them either
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # fit only on train, transform both

print("\nFeature scaling applied (StandardScaler)")
print(f"Features used: {list(X.columns)}")

## Task 5 — Performance Evaluation (15 Marks)

In [ ]:
print("=" * 50)
print("         TASK 4: MODEL BUILDING")
print("=" * 50)

# Using 7 classifiers — covering linear, tree-based, distance-based, and probabilistic
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(max_depth=8, min_samples_split=20, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'KNN'                 : KNeighborsClassifier(n_neighbors=7, metric='minkowski'),
    'SVM'                 : SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'Naive Bayes'         : GaussianNB()
    # Naive Bayes added as a probabilistic baseline — fast and interpretable
}

print("\nTraining models...\n")
trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print(f"  ✅ {name}")

print("\nAll models trained successfully.")

### 5.2 — Confusion Matrices

In [ ]:
print("=" * 55)
print("         TASK 5: PERFORMANCE EVALUATION")
print("=" * 55)

results = []
all_predictions = {}

for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    all_predictions[name] = (y_pred, y_prob)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)

    # 5-fold cross-validation score for robustness check
    cv_score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='f1').mean()

    results.append({
        'Algorithm'   : name,
        'Accuracy'    : round(acc, 4),
        'Precision'   : round(prec, 4),
        'Recall'      : round(rec, 4),
        'F1 Score'    : round(f1, 4),
        'ROC-AUC'     : round(auc, 4),
        'CV F1 (5-fold)': round(cv_score, 4)
    })

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False).reset_index(drop=True)

print("\n📊 Performance Comparison Table:\n")
print(results_df.to_string(index=False))

best_model = results_df.iloc[0]['Algorithm']
print(f"\n🏆 Best model by F1 Score: {best_model}")

### 5.3 — ROC Curve Comparison

In [ ]:
# Plotting confusion matrices for all 7 models
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold')
axes = axes.flatten()

for idx, (name, (y_pred, _)) in enumerate(all_predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['<=50K', '>50K'])
    disp.plot(ax=axes[idx], colorbar=False, cmap='Blues')
    axes[idx].set_title(name, fontsize=10)

# Hide the last empty subplot
axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

### 5.4 — Feature Importance (Random Forest)

In [ ]:
# ROC curves show the trade-off between sensitivity and specificity
# A higher AUC = better at distinguishing between the two classes
plt.figure(figsize=(10, 7))

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2', '#937860', '#DA8BC3']
for (name, (_, y_prob)), color in zip(all_predictions.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_val:.3f})", color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier (AUC = 0.5)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

### ✅ Final Summary

In [ ]:
# Random Forest gives us feature importances — useful to understand what drives income prediction
rf_model = trained_models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.plot(kind='bar', color='#4C72B0', edgecolor='black', alpha=0.85)
plt.title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.ylabel('Importance Score')
plt.xlabel('Feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

print("\nTop 5 most important features:")
print(importances.head(5).round(4))
# Relationship, marital_status, and age tend to dominate — aligns with domain intuition

In [ ]:
print("\n" + "=" * 55)
print("               FINAL SUMMARY")
print("=" * 55)
print(f"\n  Dataset size (after cleaning) : {df.shape[0]:,} rows")
print(f"  Features used                 : {X.shape[1]}")
print(f"  Models trained                : {len(trained_models)}")
print(f"  Train/Test split              : 80% / 20%")
print(f"  Scaling                       : StandardScaler")
print(f"\n  Best Model  → {results_df.iloc[0]['Algorithm']}")
print(f"  F1 Score    → {results_df.iloc[0]['F1 Score']}")
print(f"  ROC-AUC     → {results_df.iloc[0]['ROC-AUC']}")
print(f"  CV F1 Score → {results_df.iloc[0]['CV F1 (5-fold)']}")
print("\n  Saved outputs:")
print("    → eda_overview.png")
print("    → outliers_boxplot.png")
print("    → correlation_heatmap.png")
print("    → confusion_matrices.png")
print("    → roc_curves.png")
print("    → feature_importance.png")
print("=" * 55)